<a href="https://colab.research.google.com/github/jatinkumar-creator/ForensicAI/blob/main/notebooks/02_sha256_integrity_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Recreate synthetic evidence
from pathlib import Path

BASE_DIR = Path("/content/forensicai_lab")
SAMPLE_DIR = BASE_DIR / "samples"
MANIFEST_DIR = BASE_DIR / "manifests"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

samples = {
    "image1.jpg": bytes.fromhex("FF D8 FF") + b"SYNTHETIC JPEG DATA",
    "document.pdf": b"%PDF-" + b"SYNTHETIC PDF DATA",
    "image2.png": bytes.fromhex(
        "89 50 4E 47 0D 0A 1A 0A"
    ) + b"SYNTHETIC PNG DATA",
    "database.db": b"SQLite format 3\x00" + b"SYNTHETIC DB DATA",
    "archive.zip": bytes.fromhex("50 4B 03 04") + b"SYNTHETIC ZIP DATA",
    "unknown.bin": b"UNKNOWN FILE CONTENT"
}

for filename, content in samples.items():
    path = SAMPLE_DIR / filename

    if not path.exists():
        path.write_bytes(content)

print("Evidence samples ready.")
print("Number of files:", len(list(SAMPLE_DIR.iterdir())))

Evidence samples ready.
Number of files: 6


In [2]:
#Import hashlib
import hashlib

print("SHA-256 hashing module ready.")

SHA-256 hashing module ready.


In [3]:
#Build the SHA-256 hashing function
def calculate_sha256(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()

In [4]:
#Calculate hashes for all evidence files
import pandas as pd

hash_results = []

for file_path in sorted(SAMPLE_DIR.iterdir()):

    if file_path.is_file():

        file_hash = calculate_sha256(file_path)

        hash_results.append({
            "Filename": file_path.name,
            "File Size (Bytes)": file_path.stat().st_size,
            "SHA256": file_hash
        })

hash_df = pd.DataFrame(hash_results)

display(hash_df)

,Filename,File Size (Bytes),SHA256
0,archive.zip,22,4893e91e3493c8e5a28afa59f7be81fb8de4b448502afe...
1,database.db,33,d875edbbfc5dfe54f0b758453457f052d29e36e058f51f...
2,document.pdf,23,e05872ef12dbbf8d3be481fbc39fbbd76942e6a4468f72...
3,image1.jpg,22,32fe47a437f7fa097c8ee4b25f591d7f98e289c772f8bb...
4,image2.png,26,28ab5d4587196d17d5ca65dae46edbc6f25a29c7868229...
5,unknown.bin,20,35ab86af289bd3390a9b00233ed61893785f1e05a8f838...


In [5]:
#Create an evidence hash manifest
import json
from datetime import datetime, timezone

manifest = {
    "project": "ForensicAI",
    "manifest_version": "1.0",
    "hash_algorithm": "SHA-256",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "evidence": []
}

for file_path in sorted(SAMPLE_DIR.iterdir()):

    if file_path.is_file():

        manifest["evidence"].append({
            "filename": file_path.name,
            "file_size": file_path.stat().st_size,
            "sha256": calculate_sha256(file_path)
        })

manifest_path = MANIFEST_DIR / "evidence_manifest.json"

with open(manifest_path, "w") as file:
    json.dump(manifest, file, indent=4)

print("Evidence manifest created:", manifest_path)

Evidence manifest created: /content/forensicai_lab/manifests/evidence_manifest.json


In [6]:
#Build the integrity verification function
def verify_integrity(file_path, original_hash):

    current_hash = calculate_sha256(file_path)

    if current_hash == original_hash:
        return {
            "status": "MATCH",
            "original_hash": original_hash,
            "current_hash": current_hash
        }

    return {
        "status": "MISMATCH",
        "original_hash": original_hash,
        "current_hash": current_hash
    }

In [7]:
#Verify the original evidence files
verification_results = []

for record in manifest["evidence"]:

    file_path = SAMPLE_DIR / record["filename"]

    result = verify_integrity(
        file_path,
        record["sha256"]
    )

    verification_results.append({
        "Filename": record["filename"],
        "Status": result["status"]
    })

verification_df = pd.DataFrame(verification_results)

display(verification_df)

,Filename,Status
0,archive.zip,MATCH
1,database.db,MATCH
2,document.pdf,MATCH
3,image1.jpg,MATCH
4,image2.png,MATCH
5,unknown.bin,MATCH


In [8]:
#Simulate evidence modification
original_file = SAMPLE_DIR / "document.pdf"

tampered_file = BASE_DIR / "document_modified.pdf"

# Create a separate copy
tampered_file.write_bytes(original_file.read_bytes())

# Record the original hash from the manifest
original_hash = next(
    item["sha256"]
    for item in manifest["evidence"]
    if item["filename"] == "document.pdf"
)

# Modify the separate copy
with open(tampered_file, "ab") as file:
    file.write(b" MODIFIED CONTENT")

# Verify the modified copy against the original hash
result = verify_integrity(
    tampered_file,
    original_hash
)

print("Integrity Status:", result["status"])
print("Original Hash:", result["original_hash"])
print("Current Hash:", result["current_hash"])

Integrity Status: MISMATCH
Original Hash: e05872ef12dbbf8d3be481fbc39fbbd76942e6a4468f7214b35347652f03d3dd
Current Hash: 2bcc5d0436c85e40d8b2494e0c2b4ff182ca6d090fb843943c485bf68b02ad59


In [9]:
#Automated integrity tests
# Test 1: Original evidence should match
for record in manifest["evidence"]:

    file_path = SAMPLE_DIR / record["filename"]

    result = verify_integrity(
        file_path,
        record["sha256"]
    )

    assert result["status"] == "MATCH"


# Test 2: Modified copy should not match
tampered_result = verify_integrity(
    tampered_file,
    original_hash
)

assert tampered_result["status"] == "MISMATCH"

print("All SHA-256 integrity tests passed!")

All SHA-256 integrity tests passed!
